### Processing of the geojson for RAG

In [2]:
import os

os.chdir("../..")

os.getcwd()

'/Users/tombramwell/Documents/Kelder App/kelder_api/kelder_api'

In [3]:
import json

from shapely.geometry import shape

with open("src/kelder_api/assets/seamarks_and_coastlines_solent.geojson") as map:
    raw_map = json.load(map)

In [4]:
raw_map["features"][0]["geometry"]
print(f"Running {raw_map['features'][0]['properties']['name']}")
coords_list = raw_map["features"][0]["geometry"]["coordinates"][1]

feature = raw_map["features"][0]

geometry = shape(feature["geometry"])

centroid = geometry.centroid
lat = centroid.y
lon = centroid.x

print(f"Latitude: {lat}")
print(f"Longitude: {lon}")

print(f"coords: {lat},{lon}")

Running Port Solent
Latitude: 50.842721425838484
Longitude: -1.0995974549189234
coords: 50.842721425838484,-1.0995974549189234


### Process the habours

seamark:type = habour

In [18]:
def process_harbour(feature: dict) -> dict:
    # print(feature)
    try:
        name = feature["properties"]["name"]
    except KeyError:
        name = None
    # get the lat and long
    geometry = shape(feature["geometry"])

    centroid = geometry.centroid
    latitude = centroid.y
    longitude = centroid.x

    try:
        type = feature["properties"]["seamark:harbour:category"]
    except KeyError:
        type = None

    return {"name": name, "type": type, "coordinates": [longitude, latitude]}

### seamark type: fairway

"seamark:type": "fairway"

In [19]:
def process_fairway(feature: dict) -> dict:
    try:
        name = feature["properties"]["name"]
    except KeyError:
        name = None

    coordinates = feature["geometry"]["coordinates"]

    return {"name": name, "type": "fairway", "coordinates": coordinates}

### Precautionary area:
"seamark:type": "precautionary_area"

In [20]:
def process_precautionary_area(feature: dict) -> dict:
    try:
        name = feature["properties"]["seamark:name"]
    except KeyError:
        name = None

    coordinates = feature["geometry"]["coordinates"]

    return {"name": name, "type": "precautionary_area", "coordinates": coordinates}

### Lateral markers

"seamark:type": "buoy_lateral"

In [21]:
def process_lateral_marks(feature: dict, seamark_type: str) -> dict:
    try:
        name = feature["properties"]["seamark:name"]
    except KeyError:
        name = None

    coordinates = feature["geometry"]["coordinates"]

    try:
        catagory = feature["properties"][f"seamark:{seamark_type}:category"]
    except KeyError:
        colour = feature["properties"][f"seamark:{seamark_type}:colour"]
        catagory = "port" if colour == "red" else "green"

    try:
        light_character = feature["properties"]["seamark:light:character"]
        light_colour = feature["properties"]["seamark:light:colour"]
        light_group = feature["properties"]["seamark:light:group"]
        light_period = feature["properties"]["seamark:light:period"]

        light_characteristic = f"{light_character}-{light_colour}-group:{light_group}-period:{light_period}"

    except KeyError:
        light_characteristic = None

    return {
        "name": name,
        "type": "lateral_mark",
        "coordinates": coordinates,
        "catagory": catagory,
        "light": light_characteristic,
    }

### Cardinal marks

"seamark:type": "buoy_cardinal"

In [22]:
def process_cardinal_marks(feature: dict, seamark_type: str) -> dict:
    try:
        name = feature["properties"]["seamark:name"]
    except KeyError:
        name = None

    coordinates = feature["geometry"]["coordinates"]

    catagory = feature["properties"][f"seamark:{seamark_type}:category"]

    try:
        light_character = feature["properties"]["seamark:light:character"]
        light_colour = feature["properties"]["seamark:light:colour"]
        light_group = feature["properties"]["seamark:light:group"]
        light_period = feature["properties"]["seamark:light:period"]

        light_characteristic = f"{light_character}-{light_colour}-group:{light_group}-period:{light_period}"

    except KeyError:
        light_characteristic = None

    return {
        "name": name,
        "type": "cardinal_mark",
        "coordinates": coordinates,
        "catagory": catagory,
        "light": light_characteristic,
    }

### Special purpose
"seamark:type": "buoy_special_purpose"

In [23]:
def process_special_purpose(feature: dict) -> dict:
    try:
        name = feature["properties"]["seamark:name"]
    except KeyError:
        name = None

    coordinates = feature["geometry"]["coordinates"]

    try:
        light_character = feature["properties"]["seamark:light:character"]
        light_colour = feature["properties"]["seamark:light:colour"]
        light_group = feature["properties"]["seamark:light:group"]
        light_period = feature["properties"]["seamark:light:period"]

        light_characteristic = f"{light_character}-{light_colour}-group:{light_group}-period:{light_period}"

    except KeyError:
        light_characteristic = None

    return {
        "name": name,
        "type": "special_purpose",
        "coordinates": coordinates,
        "light": light_characteristic,
    }

### Isolated danger marks

"seamark:type": "buoy_isolated_danger"

In [24]:
def process_isolated_danger(feature: dict) -> dict:
    try:
        name = feature["properties"]["seamark:name"]
    except KeyError:
        name = None

    coordinates = feature["geometry"]["coordinates"]

    return {"name": name, "type": "isolated_danger", "coordinates": coordinates}

In [33]:
features = raw_map["features"]
cleaned_features = []


for n in range(0, len(features)):
    feature = features[n]

    try:
        seamark_type = feature["properties"]["seamark:type"]
    except KeyError:
        pass
    else:
        match seamark_type:
            case "harbour":
                cleaned_features.append(process_harbour(feature))
            # Commented out because its a linestring
            # case "fairway":
            #     cleaned_features.append(process_fairway(feature))
            # case "precautionary_area":
            #     cleaned_features.append(process_precautionary_area(feature))
            case "buoy_lateral" | "beacon_lateral":
                cleaned_features.append(process_lateral_marks(feature, seamark_type))
            case "buoy_cardinal" | "beacon_cardinal":
                cleaned_features.append(process_cardinal_marks(feature, seamark_type))
            case "buoy_special_purpose" | "beacon_special_purpose":
                cleaned_features.append(process_special_purpose(feature))
            case "buoy_isolated_danger" | "beacon_isolated_danger":
                cleaned_features.append(process_isolated_danger(feature))

In [34]:
print(len(cleaned_features))

print(cleaned_features[-1])

943
{'name': None, 'type': 'lateral_mark', 'coordinates': [-1.0226469, 50.8141227], 'catagory': 'starboard', 'light': None}


In [57]:
with open("src/kelder_api/assets/marks.json", "w") as f:
    json.dump(cleaned_features, f, indent=2)

# Consider some searching and querying options

rtree

In [49]:
from rtree import index
from shapely.geometry import Point

idx = index.Index()
id_counter = 0

for mark in cleaned_features:
    lon, lat = mark["coordinates"]
    point = Point(lon, lat)

    idx.insert(id_counter, point.bounds, obj=mark)
    id_counter += 1

Query the index spacially 

In [50]:
query_point = (-1.09, 50.8432)
search_box = (query_point[0], query_point[1], query_point[0], query_point[1])

result = list(idx.nearest(search_box, 10, objects=True))[0]

mark = result.object  # <-- this is your original dict
print(mark)

{'name': 'Port Solent', 'type': 'marina', 'coordinates': [-1.0995974549189234, 50.842721425838484]}


In [51]:
my_position = (-1.0226469, 50.8141227)

# Create a zero-area bounding box for the point
bbox = (my_position[0], my_position[1], my_position[0], my_position[1])

nearest = list(idx.nearest(bbox, 10, objects=True))
print("Nearest mark:", nearest[0].object)
print("Nearest mark:", nearest[9].object)

Nearest mark: {'name': None, 'type': 'lateral_mark', 'coordinates': [-1.0226469, 50.8141227], 'catagory': 'starboard', 'light': None}
Nearest mark: {'name': 'West Sword', 'type': 'lateral_mark', 'coordinates': [-1.0294962, 50.8081653], 'catagory': 'starboard', 'light': None}


Retrieve all the named marks

In [55]:
names = ", ".join([mark["name"] for mark in cleaned_features if mark["name"] != None])

# Process the coastline:

Southampton water: Q502233
The solent: 4095105
River itchen: 3534881
The river test: 3535360
The hamble: 3518905
